In [ ]:
from pathlib import Path
import numpy as np
import io, time, threading, queue

import imageio.v2 as imageio
from tqdm.auto import tqdm
from astropy.io import fits
from astroquery.mast import Observations

from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display

# ---------------------------
# WHAT YOU WANT TO SEE
# ---------------------------
TARGET_NAME = "Mars"
INSTRUMENT  = "NIRCAM"
RADIUS      = "0.3 deg"

# Webb's first Mars NIRCam images (Sept 5, 2022) were GTO Program 1415.
# Set to None to search all programs instead.
PROGRAM_ID = 1415  # or None

PREFER_KEYS = ("CALINTS", "RATEINTS", "CAL", "RATE", "I2D")

MAX_OBS_TO_TRY   = 30
MAX_PRODUCTS_TO_TRY_PER_OBS = 6
MAX_FRAMES_TOTAL = 400

DO_ALIGN = True          # FFT phase-correlation; set False if too slow
FPS_VIDEO   = 12
FPS_PREVIEW = 12
PREVIEW_BUFFER = 180     # loop last N frames while still working

OUT_DIR = Path("jwst_mars_video")
DATA_DIR = OUT_DIR / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

video_path = OUT_DIR / f"MARS_{INSTRUMENT}_preview.mp4"
print("Will write:", video_path.resolve())

# ---------------------------
# Helpers (mostly identical)
# ---------------------------
def to_frame_cube(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr)
    if a.ndim == 2:
        return a[None, :, :]
    if a.ndim == 3:
        # typical: (nframes, y, x)
        if a.shape[0] <= 10000:
            return a
        # sometimes: (y, x, nframes)
        if a.shape[2] <= 10000:
            return np.moveaxis(a, 2, 0)
        raise ValueError(f"Unclear 3D layout: {a.shape}")
    if a.ndim == 4:
        # (nints, ngroups, y, x) -> average groups
        return np.nanmean(a, axis=1)
    raise ValueError(f"Unsupported shape: {a.shape}")

def subgroup_col(tbl):
    for c in tbl.colnames:
        if c.lower() in ("productsubgroupdescription", "productsubgroupdesc"):
            return c
    return None

def normalize_target_str(s: str) -> str:
    return (s or "").strip().upper()

def filter_to_target(obs_tbl, target_name: str):
    tgt = normalize_target_str(target_name)
    for col in ("target_name", "target", "targname", "objname"):
        if col in obs_tbl.colnames:
            vals = np.char.upper(obs_tbl[col].astype(str))
            m = np.char.find(vals, tgt) >= 0
            if np.any(m):
                return obs_tbl[m]
    # If no obvious target column, return as-is
    return obs_tbl

def get_obs_table(target, instrument, radius, program_id=None):
    if program_id is None:
        obs = Observations.query_object(target, radius=radius)
    else:
        obs = Observations.query_criteria(obs_collection="JWST", proposal_id=program_id)
        obs = filter_to_target(obs, target)

    obs = obs[obs["obs_collection"] == "JWST"]
    inst = np.char.upper(obs["instrument_name"].astype(str))
    obs = obs[np.char.find(inst, instrument.upper()) >= 0]

    # Prefer longer/denser observations when available
    if "t_exptime" in obs.colnames:
        obs.sort("t_exptime")
        obs = obs[::-1]
    elif "t_min" in obs.colnames:
        obs.sort("t_min")
        obs = obs[::-1]
    return obs

def pick_products_for_obs(obs_row, prefer_keys=PREFER_KEYS):
    products = Observations.get_product_list(obs_row)
    p = Observations.filter_products(products, productType="SCIENCE", extension="fits")
    if len(p) == 0:
        return None

    sc = subgroup_col(p)
    fn = np.char.lower(p["productFilename"].astype(str))

    for key in prefer_keys:
        if sc and np.any(p[sc] == key):
            return p[p[sc] == key]
        if np.any(np.char.find(fn, key.lower()) >= 0):
            return p[np.char.find(fn, key.lower()) >= 0]

    return p

def download_one(product_row, download_dir):
    manifest = Observations.download_products(product_row[0:1], download_dir=str(download_dir), cache=True)
    local_col = "Local Path" if "Local Path" in manifest.colnames else "local_path"
    return Path(manifest[local_col][0])

def read_frames(fits_path: Path):
    with fits.open(fits_path) as hdul:
        if "SCI" not in hdul:
            raise RuntimeError("No SCI extension.")
        data = hdul["SCI"].data
    return to_frame_cube(data).astype(np.float32)

def background_subtract(frame):
    return frame - np.nanmedian(frame)

def find_disk_bbox(frame, pad=30):
    m = np.nan_to_num(frame, nan=np.nanmedian(frame))
    thr = np.nanpercentile(m, 90)
    mask = m > thr
    if not np.any(mask):
        h, w = m.shape
        return 0, h, 0, w
    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    h, w = m.shape
    y0 = max(0, y0 - pad); y1 = min(h, y1 + pad)
    x0 = max(0, x0 - pad); x1 = min(w, x1 + pad)
    return y0, y1, x0, x1

def robust_vmin_vmax(frames3d):
    vmin, vmax = np.nanpercentile(frames3d, [2, 99.7])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = np.nanmin(frames3d), np.nanmax(frames3d)
    return float(vmin), float(vmax)

def scale_to_uint8(img, vmin, vmax):
    x = np.nan_to_num(img, nan=vmin)
    x = np.clip((x - vmin) / (vmax - vmin + 1e-12), 0, 1)
    x = np.arcsinh(10 * x) / np.arcsinh(10)
    return (255 * x).astype(np.uint8)

def annotate(rgb, text):
    im = Image.fromarray(rgb)
    draw = ImageDraw.Draw(im)
    draw.rectangle([0, 0, im.size[0], 26], fill=(0, 0, 0))
    draw.text((6, 5), text, fill=(255, 255, 255))
    return np.array(im)

def phase_corr_shift(ref, img):
    eps = 1e-9
    F = np.fft.fft2(ref)
    G = np.fft.fft2(img)
    R = F * np.conj(G)
    R /= (np.abs(R) + eps)
    corr = np.abs(np.fft.ifft2(R))
    y, x = np.unravel_index(np.argmax(corr), corr.shape)
    if y > corr.shape[0] // 2: y -= corr.shape[0]
    if x > corr.shape[1] // 2: x -= corr.shape[1]
    return int(y), int(x)

def align_to_reference(frames3d):
    ref = frames3d[0]
    out = [ref]
    for i in range(1, len(frames3d)):
        dy, dx = phase_corr_shift(ref, frames3d[i])
        out.append(np.roll(frames3d[i], shift=(dy, dx), axis=(0, 1)))
    return np.stack(out, axis=0)

def rgb_png_bytes(rgb):
    buf = io.BytesIO()
    Image.fromarray(rgb).save(buf, format="PNG")
    return buf.getvalue()

# ---------------------------
# Live preview widget thread
# ---------------------------
img_widget = widgets.Image(format="png")
display(img_widget)

frame_q = queue.Queue(maxsize=PREVIEW_BUFFER)
stop_flag = {"stop": False}

def preview_worker():
    last = []
    i = 0
    while not stop_flag["stop"]:
        try:
            b = frame_q.get(timeout=0.2)
            last.append(b)
            if len(last) > PREVIEW_BUFFER:
                last = last[-PREVIEW_BUFFER:]
            img_widget.value = b
        except queue.Empty:
            # loop the last frames
            if last:
                img_widget.value = last[i % len(last)]
                i += 1
                time.sleep(1 / FPS_PREVIEW)

t = threading.Thread(target=preview_worker, daemon=True)
t.start()

# ---------------------------
# Main: find a good cube, render, write mp4
# ---------------------------
obs = get_obs_table(TARGET_NAME, INSTRUMENT, RADIUS, PROGRAM_ID)
print(f"Found {len(obs)} JWST obs rows matching {TARGET_NAME}/{INSTRUMENT}.")

best = None  # (nframes, fits_path, frames3d)
for obs_row in obs[:MAX_OBS_TO_TRY]:
    prods = pick_products_for_obs(obs_row)
    if prods is None or len(prods) == 0:
        continue

    # try larger files first (when size exists)
    if "size" in prods.colnames:
        prods.sort("size")
        prods = prods[::-1]

    tried = 0
    for pr in prods[:MAX_PRODUCTS_TO_TRY_PER_OBS]:
        tried += 1
        try:
            fp = download_one(pr, DATA_DIR)
            fr = read_frames(fp)
            n = fr.shape[0]
            if best is None or n > best[0]:
                best = (n, fp, fr)
            # stop early if we got something "movie-like"
            if n >= 20:
                break
        except Exception as e:
            # skip bad/odd products
            continue

    if best is not None and best[0] >= 20:
        break

if best is None:
    stop_flag["stop"] = True
    raise RuntimeError("Could not find a usable Mars NIRCam FITS cube (SCI) from the selected search.")

nframes, fits_path, frames = best
print(f"Using: {fits_path.name} with {nframes} frame(s).")

# limit frames for memory/time
if nframes > MAX_FRAMES_TOTAL:
    idx = np.linspace(0, nframes - 1, MAX_FRAMES_TOTAL).round().astype(int)
    frames = frames[idx]
    nframes = len(frames)

# preprocess: background-subtract + crop around disk using median frame
med = np.nanmedian(frames, axis=0)
y0, y1, x0, x1 = find_disk_bbox(med, pad=40)

frames_bs = np.array([background_subtract(f)[y0:y1, x0:x1] for f in frames], dtype=np.float32)

# optional alignment on the cropped frames
if DO_ALIGN and len(frames_bs) > 1:
    frames_bs = align_to_reference(frames_bs)

vmin, vmax = robust_vmin_vmax(frames_bs)

# write mp4
with imageio.get_writer(video_path, fps=FPS_VIDEO) as writer:
    for i in tqdm(range(nframes), desc="Rendering"):
        g = scale_to_uint8(frames_bs[i], vmin, vmax)
        rgb = np.stack([g, g, g], axis=-1)
        label = f"Mars | {INSTRUMENT} | {i+1}/{nframes} | {fits_path.name}"
        rgb = annotate(rgb, label)

        # push to live preview (non-blocking)
        try:
            frame_q.put_nowait(rgb_png_bytes(rgb))
        except queue.Full:
            pass

        writer.append_data(rgb)

stop_flag["stop"] = True
print("Done! Video written to:", video_path.resolve())


Will write: /home/hw1970218/Desktop/fits2/jwst_mars_video/MARS_NIRCAM_preview.mp4


Image(value=b'')

Found 2 JWST obs rows matching Mars/NIRCAM.